# Luminancia

## Media de luminancia

Promedia los valores de luminancia de todos los píxeles para obtener el nivel de brillo general de la imagen.

In [1]:
from skimage import io
import numpy as np

def obtener_media_luminancia(img_path):
    return np.mean(io.imread(img_path, as_gray=True))

## Varianza de luminancia

Mide qué tan uniformemente distribuida está la luminancia. Valores altos = alto contraste; valores bajos = uniformidad.

In [43]:
from skimage import io
import numpy as np

def obtener_varianza_luminancia(img_path):
    return np.var(io.imread(img_path, as_gray=True))

## Rango dinámico

Diferencia entre el valor máximo y mínimo de luminancia. Indica qué tan bien se captura el espectro completo de brillos.

In [44]:
from skimage import io
import numpy as np
from skimage.util import img_as_float


def obtener_rango_dinamico(img_path, epsilon=1e-6):
    img = img_as_float(io.imread(img_path, as_gray=True))
    y_min = np.min(img)
    y_max = np.max(img)

    return y_max / (y_min + epsilon)

## Desviación estándar de brillo

Raíz cuadrada de la varianza de luminancia. Representa el contraste global típico de la imagen.

In [45]:
from skimage import io
import numpy as np

def obtener_desv_estandar_brillo(img_path):
    return np.std(io.imread(img_path, as_gray=True))

# Contraste 

## Contraste de Michelson

Contraste modular entre máximo y mínimo de luminancia en una región. Adimensional (0 a 1), robusto ante cambios globales de brillo.

In [46]:
from skimage import io
import numpy as np

def obtener_contraste_michelson(img_path):
    img = io.imread(img_path, as_gray=True)
    
    Lmax = np.percentile(img, 98)
    Lmin = np.percentile(img, 2)
    
    return (Lmax - Lmin) / (Lmax + Lmin + 1e-8)

##  Contraste basado en gradiente 
Calcula el contraste basado en gradiente de una imagen,
usando la magnitud media del gradiente de Sobel.


In [47]:
from skimage import io, filters
from skimage.util import img_as_float
import numpy as np

def obtener_contraste_gradiente(img_path):
   
    img = img_as_float(io.imread(img_path, as_gray=True))
    gradiente = filters.sobel(img)
    return np.mean(gradiente)

# Entropía (Contenido de Información)


## Entropía de Shannon

Mide la incertidumbre o cantidad de información promedio en la distribución de intensidades. Indicador de complejidad.

In [48]:
from skimage import io
from skimage.measure import shannon_entropy
from skimage.util import img_as_ubyte


def obtener_entropia_shannon(img_path):
    img = img_as_ubyte(io.imread(img_path, as_gray=True))
    return shannon_entropy(img)

## Entropía espectral
Calcula la entropía espectral de una imagen, midiendo cómo se
    distribuye la potencia espectral en el dominio de la frecuencia
    (Shannon, 1948).

In [49]:
from skimage import io
from skimage.util import img_as_float
import numpy as np
from scipy.stats import entropy

def obtener_entropia_espectral(img_path):
    img = img_as_float(io.imread(img_path, as_gray=True))

    # Transformada de Fourier 2D (rfft2 es más eficiente para entradas reales)
    fft = np.fft.rfft2(img)

    # Densidad espectral de potencia: magnitud al cuadrado
    S = np.abs(fft) ** 2

    # Normalización a distribución de probabilidad y cálculo de entropía
    P = S.ravel() / S.sum()

    return entropy(P, base=2)

## Entropía de gradiente
Calcula la entropía de gradiente (H_G) de una imagen, aplicando
    la entropía de Shannon sobre la distribución de magnitudes del
    gradiente (Jiang & Shang, 2025).

In [50]:
from skimage import io, filters
from skimage.util import img_as_float
import numpy as np
from scipy.stats import entropy

def obtener_entropia_gradiente(img_path, bins=256):
    img = img_as_float(io.imread(img_path, as_gray=True))

    # Magnitud del gradiente mediante el operador de Sobel
    magnitud_gradiente = filters.sobel(img)

    # Histograma de magnitudes de gradiente, normalizado a probabilidad
    hist, _ = np.histogram(magnitud_gradiente, bins=bins, range=(0, magnitud_gradiente.max() + 1e-12))
    p = hist / hist.sum()

    return entropy(p, base=2)

# Frecuencias

## Energía en altas frecuencias

Concentración de potencia espectral en componentes de alta frecuencia. Indica presencia de bordes y detalles nítidos.

In [51]:
from skimage import io
from skimage.util import img_as_float
from scipy.fft import fft2, fftshift
import numpy as np

def obtener_energia_altas_frecuencias(img_path, radio=0.5):
    img = img_as_float(io.imread(img_path, as_gray=True))
    F = fftshift(fft2(img))

    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]

    radio_corte = radio * min(M, N) / 2
    mask = (X - cx) ** 2 + (Y - cy) ** 2 > radio_corte ** 2

    energia_total = np.sum(np.abs(F) ** 2)
    if energia_total == 0:
        return 0.0

    energia_alta = np.sum(np.abs(F[mask]) ** 2)

    return energia_alta / energia_total

## Energía en bajas frecuencias

Potencia espectral en componentes de baja frecuencia. Corresponde a variaciones suaves, fondos y formas generales.

In [52]:
from skimage import io
from skimage.util import img_as_float
from scipy.fft import fft2, fftshift
import numpy as np

def obtener_energia_bajas_frecuencias(img_path, radio=0.5):
    img = img_as_float(io.imread(img_path, as_gray=True))
    F = fftshift(fft2(img))

    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]

    radio_corte = radio * min(M, N) / 2
    mask = (X - cx) ** 2 + (Y - cy) ** 2 <= radio_corte ** 2

    energia_total = np.sum(np.abs(F) ** 2)
    if energia_total == 0:
        return 0.0

    energia_baja = np.sum(np.abs(F[mask]) ** 2)

    return energia_baja / energia_total

## Relación high/low frequency ratio

Proporción de energía entre altas y bajas frecuencias. Evalúa el nivel de detalle o nitidez frente a la estructura global.

In [53]:
from skimage import io
from skimage.util import img_as_float
from scipy.fft import fft2, fftshift
import numpy as np


def obtener_hlfr(img_path, radio=0.5, epsilon=1e-6):
    img = img_as_float(io.imread(img_path, as_gray=True))
    F = fftshift(fft2(img))

    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]

    radio_corte = radio * min(M, N) / 2
    dist2 = (X - cx) ** 2 + (Y - cy) ** 2

    mask_alta = dist2 > radio_corte ** 2
    mask_baja = ~mask_alta

    energia_alta = np.sum(np.abs(F[mask_alta]) ** 2)
    energia_baja = np.sum(np.abs(F[mask_baja]) ** 2)

    return energia_alta / (energia_baja + epsilon)

# Bordes y Gradientes

## Densidad de bordes

Fracción de píxeles detectados como bordes respecto al total de píxeles de la imagen.

In [54]:
from skimage import io, feature
from skimage.util import img_as_float
import numpy as np

def obtener_densidad_bordes(img_path, sigma=1.0):
    img = img_as_float(io.imread(img_path, as_gray=True))
    mapa_bordes = feature.canny(img, sigma=sigma)

    M, N = mapa_bordes.shape
    n_e = np.count_nonzero(mapa_bordes)

    return n_e / (M * N)

## Desviación estándar del gradiente

Variabilidad de la magnitud del gradiente. Cuantifica la complejidad estructural de la imagen.

In [55]:
from skimage import io, filters
from skimage.util import img_as_float
import numpy as np

def obtener_desv_estandar_gradiente(img_path):
    img = img_as_float(io.imread(img_path, as_gray=True))
    magnitud = filters.sobel(img)
    return np.std(magnitud)

## Densidad de bordes
 Calcula la densidad de bordes de una imagen, definida como la
    proporción de píxeles identificados como borde respecto del
    total de píxeles de la imagen (Sadykova & James, 2017).

In [56]:
from skimage import io, feature
from skimage.util import img_as_float
import numpy as np

def obtener_densidad_bordes(img_path, sigma=1.0):
    img = img_as_float(io.imread(img_path, as_gray=True))
    mapa_bordes = feature.canny(img, sigma=sigma, low_threshold=0.1, high_threshold=0.2)

    M, N = mapa_bordes.shape
    n_e = np.count_nonzero(mapa_bordes)

    return n_e / (M * N)



# Textura

## Homogeneidad de textura (GLCM)

Qué tan cercanos son los valores de intensidad de píxeles vecinos. Alto = textura suave o uniforme.

In [57]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_homogeneidad_textura(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    return graycoprops(glcm, 'homogeneity')[0, 0]

## Energía (GLCM Energy / ASM)

Mide la uniformidad global (Angular Second Moment). Alto = textura uniforme o repetitiva.

In [59]:
from skimage import io
from skimage.util import img_as_ubyte
from skimage.feature import graycomatrix, graycoprops
import numpy as np


def obtener_energia_glcm(img_path, distancias=[1], angulos=[0], niveles=256):
    img = img_as_ubyte(io.imread(img_path, as_gray=True))

    glcm = graycomatrix(
        img,
        distances=distancias,
        angles=angulos,
        levels=niveles,
        symmetric=True,
        normed=True
    )

    energy = float(np.mean(graycoprops(glcm, prop='energy')))
    

    return energy

## Correlación (GLCM)

Dependencia lineal entre niveles de gris de píxeles vecinos. Alta correlación = patrón estructural consistente.

In [60]:
from skimage import io
from skimage.util import img_as_ubyte
from skimage.feature import graycomatrix, graycoprops

def obtener_correlacion_glcm(img_path, distancias=[1], angulos=[0], niveles=256):
    img = img_as_ubyte(io.imread(img_path, as_gray=True))

    glcm = graycomatrix(
        img,
        distances=distancias,
        angles=angulos,
        levels=niveles,
        symmetric=True,
        normed=True
    )

    correlacion = graycoprops(glcm, prop='correlation')

    return float(np.mean(correlacion))

## Contraste de textura (GLCM Contrast)

Variaciones locales de intensidad entre píxeles vecinos. Cuantifica la aspereza o definición de bordes en la textura.

In [61]:
from skimage import io
from skimage.util import img_as_ubyte
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_contraste_glcm(img_path, distancias=[1], angulos=[0], niveles=256):
    img = img_as_ubyte(io.imread(img_path, as_gray=True))

    glcm = graycomatrix(
        img,
        distances=distancias,
        angles=angulos,
        levels=niveles,
        symmetric=True,
        normed=True
    )

    contraste = graycoprops(glcm, prop='contrast')

    return float(np.mean(contraste))

## Entropía de textura (GLCM Entropy)

Grado de aleatoriedad o desorden en la distribución espacial de niveles de gris entre píxeles vecinos.

In [62]:
from skimage import io
from skimage.util import img_as_ubyte
from skimage.feature import graycomatrix
import numpy as np

def obtener_entropia_glcm(img_path, distancias=[1], angulos=[0], niveles=256):
    img = img_as_ubyte(io.imread(img_path, as_gray=True))

    glcm = graycomatrix(
        img,
        distances=distancias,
        angles=angulos,
        levels=niveles,
        symmetric=True,
        normed=True
    )

    # Promediar sobre las distancias/ángulos antes de calcular entropía
    p = glcm.mean(axis=(2, 3))

    # Evitar log2(0) excluyendo probabilidades nulas
    p_nonzero = p[p > 0]

    return float(-np.sum(p_nonzero * np.log2(p_nonzero)))

# Saturación y Crominancia

## Cargar imagen RGB
Para trabajar todas las imagenes en los 3 canales RGB

In [8]:
from skimage import io, color
from skimage.util import img_as_float
import numpy as np

def cargar_imagen_rgb(img_path):
    img = img_as_float(io.imread(img_path))

    if img.ndim == 2:
        # Escala de grises -> replicar a 3 canales
        img = color.gray2rgb(img)
    elif img.ndim == 3 and img.shape[2] == 4:
        # RGBA -> descartar canal alfa
        img = img[:, :, :3]
    elif img.ndim == 3 and img.shape[2] == 3:
        # Ya es RGB, no hacer nada
        pass
    else:
        raise ValueError(f"Formato de imagen no soportado: shape={img.shape}")

    return img

## Media de saturación

Nivel promedio de pureza cromática en la imagen en espacio HSV.

In [65]:
from skimage import  color
import numpy as np

def obtener_media_saturacion(img_path):
    img_rgb = cargar_imagen_rgb(img_path)
    img_hsv = color.rgb2hsv(img_rgb)
    saturacion = img_hsv[:, :, 1]
    return float(np.mean(saturacion))

## Varianza de saturación

Dispersión de valores de saturación. Indica la uniformidad o variabilidad cromática de la imagen.

In [9]:
from skimage import io, color
from skimage.util import img_as_float
import numpy as np


def obtener_varianza_saturacion(img_path):
    
    img_rgb = cargar_imagen_rgb(img_path)
    img_hsv = color.rgb2hsv(img_rgb)
    saturacion = img_hsv[:, :, 1]

    return float(np.var(saturacion))

## Varianza de matiz (Hue variance)

Dispersión angular de los tonos de color dominantes. Indica la diversidad cromática en el espacio HSV.

In [6]:
from scipy.stats import circvar

def obtener_varianza_matiz(img_path, umbral_saturacion=0.1):
    img_rgb = cargar_imagen_rgb(img_path)
    img_hsv = color.rgb2hsv(img_rgb)

    hue = img_hsv[:, :, 0]
    saturacion = img_hsv[:, :, 1]

    mask = saturacion > umbral_saturacion
    theta = hue[mask] * 2 * np.pi

    if theta.size == 0:
        return np.nan

    return float(circvar(theta, high=2*np.pi, low=0))

# Resolución Espacial

## Megapíxeles

Resolución total de la imagen expresada en millones de píxeles.

In [ ]:
from skimage import io

def obtener_megapixeles(img_path):
    img = io.imread(img_path)
    M, N = img.shape[0], img.shape[1]

    return (M * N) / 1_000_000

## Aspect ratio

Proporción entre el ancho y el alto de la imagen, expresada como razón W/H.

In [ ]:
from skimage import io

def obtener_aspect_ratio(img_path):
    img = io.imread(img_path)
    M, N = img.shape[:2]
    


    return N / M